In [2]:
import torch 
import torch.nn as nn
import math

class LayerNormalization(nn.Module):

    def __init__(self, features: int, eps:float=10**-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features)) # alpha is a learnable parameter
        self.bias = nn.Parameter(torch.zeros(features)) # bias is a learnable parameter

    def forward(self, x):
        # x: (batch, seq_len, hidden_size)
        # Keep the dimention for broadcasting
        mean = x.mean(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # Keep the dimention for broadcasting
        std = x.std(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # eps is to prevent dividing by zero or when std is very small
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

# Test the LayerNormalization class
if __name__ == "__main__":
    layer_norm = LayerNormalization(features=4)
    x = torch.randn(2, 3, 4)  # Example input tensor
    output = layer_norm(x)
    print("Input:\n", x)
    print("Output:\n", output)
    print("Alpha:\n", layer_norm.alpha)
    print("Bias:\n", layer_norm.bias)
    print("Output shape:", output.shape)  # Should be the same shape as input
    print("Mean of output:", output.mean(dim=-1))  # Should be close to zero
    print("Std of output:", output.std(dim=-1))  # Should be close to one
    print("Epsilon:", layer_norm.eps)  # Should be the same as defined
    print("Alpha shape:", layer_norm.alpha.shape)  # Should be (4,)
    print("Bias shape:", layer_norm.bias.shape)  # Should be (4,)
    print("LayerNorm parameters:", list(layer_norm.parameters()))  # Should show alpha and bias
    print("LayerNorm state_dict:", layer_norm.state_dict())  # Should show alpha and bias
    print("LayerNorm module:", layer_norm)  # Should show the LayerNormalization module
    print("LayerNorm module type:", type(layer_norm))  # Should be <class '__main__.LayerNormalization'>
    print("LayerNorm module class name:", layer_norm.__class__.__name__)  # Should be 'LayerNormalization'
    print("LayerNorm module class:", layer_norm.__class__)  # Should be <class '__main__.LayerNormalization'>
    print("LayerNorm module class type:", layer_norm.__class__.__module__)  # Should be '__main__'
    print("LayerNorm module class qualified name:", layer_norm.__class__.__qualname__)  # Should be 'LayerNormalization'
    print("LayerNorm module class doc:", layer_norm.__class__.__doc__)  # Should be None or the docstring of LayerNormalization
    print("LayerNorm module class str:", str(layer_norm))  # Should show the string representation of the LayerNormalization module
    print("LayerNorm module class repr:", repr(layer_norm))  # Should show the string    

Input:
 tensor([[[ 0.4030,  0.5597, -1.2076,  1.0346],
         [ 1.8776, -0.3635, -0.6705,  0.9885],
         [-1.7992, -1.9866, -0.6820,  0.7085]],

        [[ 0.9940, -0.2870, -1.0265,  0.2586],
         [ 0.3628, -0.0963, -1.2940, -0.6185],
         [ 0.1533, -1.2539, -0.9291,  0.6417]]])
Output:
 tensor([[[ 0.2110,  0.3718, -1.4419,  0.8592],
         [ 1.1934, -0.6906, -0.9487,  0.4459],
         [-0.6927, -0.8437,  0.2078,  1.3286]],

        [[ 1.1812, -0.3181, -1.1836,  0.3205],
         [ 1.0876,  0.4427, -1.2396, -0.2907],
         [ 0.5606, -1.0162, -0.6523,  1.1079]]], grad_fn=<AddBackward0>)
Alpha:
 Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)
Bias:
 Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)
Output shape: torch.Size([2, 3, 4])
Mean of output: tensor([[0.0000e+00, 7.4506e-09, 0.0000e+00],
        [0.0000e+00, 2.9802e-08, 2.9802e-08]], grad_fn=<MeanBackward1>)
Std of output: tensor([[1.0000, 1.0000, 1.0000],
        [1.0000, 1.

### Layer Normalization

Layer Normalization is applied independently to each input sample, across its feature dimension.

---

#### **Equation:**

$$
\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta
$$

---

#### **Where:**

- \( x \): Input vector of shape \([d]\) (e.g., token embedding or hidden state)  
- \( \mu \): Mean of the input vector  
  $$
  \mu = \frac{1}{d} \sum_{i=1}^{d} x_i
  $$
- \( \sigma^2 \): Variance of the input vector  
  $$
  \sigma^2 = \frac{1}{d} \sum_{i=1}^{d} (x_i - \mu)^2
  $$
- \( \epsilon \): A small constant added for numerical stability (e.g., \(10^{-5}\))  
- \( \gamma, \beta \): Learnable scale and shift parameters of shape \([d]\)

---

#### **Key Characteristics:**

- Operates across the feature dimension (not batch).
- Used extensively in Transformer-based architectures.
- Helps stabilize training by reducing internal covariate shift.
- Unlike BatchNorm, works well with variable sequence lengths and small batch sizes.

### ***L

In [3]:
torch.nn.LayerNorm(normalized_shape=4, eps=1e-05, elementwise_affine=True, dtype=None)

LayerNorm((4,), eps=1e-05, elementwise_affine=True)

In [6]:
def welford_variance(data):
    n = 0
    mean = 0.0
    M2 = 0.0
    for x in data:
        n += 1
        delta = x - mean
        mean += delta / n
        M2 += delta * (x - mean)
    population_var = M2 / n
    sample_var = M2 / (n - 1) if n > 1 else float('nan')
    return mean, population_var, sample_var

X = [2, 4, 6, 8, 10]
mean, pop_var, sample_var = welford_variance(X)
print(f"Mean: {mean}, Population Var: {pop_var}, Sample Var: {sample_var}")

Mean: 6.0, Population Var: 8.0, Sample Var: 10.0


In [1]:
import torch
import torch.nn as nn

class WelfordLayerNorm(nn.Module):

    def __init__(self, features: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features)) # alpha is a learnable parameter
        self.bias = nn.Parameter(torch.zeros(features)) # bias is a learnable parameter
        self.features = features

    def forward(self, x):
        # x: (batch, seq_len, hidden_size)
        batch_size, seq_len, _ = x.shape
        mean = torch.zeros(batch_size, seq_len, self.features, device=x.device)
        M2 = torch.zeros(batch_size, seq_len, self.features, device=x.device)
        
        for i in range(seq_len):
            xi = x[:, i, :]
            delta = xi - mean[:, i, :]
            mean[:, i, :] += delta / (i + 1)
            M2[:, i, :] += delta * (xi - mean[:, i, :])
        population_var = M2 / seq_len
        std = torch.sqrt(population_var + self.eps)
        return self.alpha * (x - mean) / std + self.bias
    
# Test the WelfordLayerNorm class
if __name__ == "__main__":  
    welford_layer_norm = WelfordLayerNorm(features=4)
    x = torch.randn(2, 3, 4)  # Example input tensor
    output = welford_layer_norm(x)
    print("Input:\n", x)
    print("Output:\n", output)
    print("Alpha:\n", welford_layer_norm.alpha)
    print("Bias:\n", welford_layer_norm.bias)
    print("Output shape:", output.shape)  # Should be the same shape as input
    print("Mean of output:", output.mean(dim=-1))  # Should be close to zero
    print("Std of output:", output.std(dim=-1))  # Should be close to one
    print("Epsilon:", welford_layer_norm.eps)  # Should be the same as defined
    print("Alpha shape:", welford_layer_norm.alpha.shape)  # Should be (4,)
    print("Bias shape:", welford_layer_norm.bias.shape)  # Should be (4,)
    print("WelfordLayerNorm parameters:", list(welford_layer_norm.parameters()))  # Should show alpha and bias
    print("WelfordLayerNorm state_dict:", welford_layer_norm.state_dict())  # Should show alpha and bias

Input:
 tensor([[[-0.2114,  1.9860,  0.2260,  1.4549],
         [-0.2148, -1.4200,  1.2465,  0.2510],
         [-0.7649, -0.4447,  0.5980, -0.5377]],

        [[-0.0359,  0.4925, -0.9039, -1.0009],
         [-1.0526,  2.4567,  1.9755, -0.8098],
         [-0.0110, -1.1834,  1.2834, -0.9569]]])
Output:
 tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000],
         [-1.2247, -1.2247,  1.2247,  1.2247],
         [-1.4142, -1.4142,  1.4142, -1.4142]],

        [[ 0.0000,  0.0000,  0.0000,  0.0000],
         [-1.2247,  1.2247,  1.2247, -1.2247],
         [-1.3886, -1.4142,  1.4142, -1.4142]]], grad_fn=<AddBackward0>)
Alpha:
 Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)
Bias:
 Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)
Output shape: torch.Size([2, 3, 4])
Mean of output: tensor([[ 0.0000e+00,  5.1558e-06, -7.0710e-01],
        [ 0.0000e+00,  1.9073e-06, -7.0069e-01]], grad_fn=<MeanBackward1>)
Std of output: tensor([[0.0000, 1.4142, 1.4142],
        [0.00

In [5]:
# compare and benchmark LayerNormalization with PyTorch's built-in LayerNorm vs welformed LayerNorm
naive_layer_norm = LayerNormalization(features=4)
pytorch_layer_norm = nn.LayerNorm(normalized_shape=4, eps=1e-05, elementwise_affine=True)
welford_layer_norm = WelfordLayerNorm(features=4)
print("Naive LayerNorm parameters:", list(naive_layer_norm.parameters()))
print("PyTorch LayerNorm parameters:", list(pytorch_layer_norm.parameters()))
print("Welford LayerNorm parameters:", list(welford_layer_norm.parameters()))

x = torch.randn(1000, 10, 4)  # Example input tensor for benchmarking
import time
start_time = time.time()
naive_output = naive_layer_norm(x)
naive_time = time.time() - start_time
print(f"Naive LayerNorm time: {naive_time:.6f} seconds")

start_time = time.time()
pytorch_output = pytorch_layer_norm(x)
pytorch_time = time.time() - start_time
print(f"PyTorch LayerNorm time: {pytorch_time:.6f} seconds")


start_time = time.time()
welford_output = welford_layer_norm(x)
welford_time = time.time() - start_time
print(f"Welford LayerNorm time: {welford_time:.6f} seconds")

Naive LayerNorm parameters: [Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True), Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)]
PyTorch LayerNorm parameters: [Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True), Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)]
Welford LayerNorm parameters: [Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True), Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)]
Naive LayerNorm time: 0.005802 seconds
PyTorch LayerNorm time: 0.004131 seconds
Welford LayerNorm time: 0.001337 seconds


In [ ]:
import torch
import torch.nn as nn 
import math
import numpy as np

class LayerNormalization(nn.Module):

    def __inti__(self, features: int, eps: float=10**-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features)) # alpha is a learnable parameter
        self.bias = nn.Parameter(torch.zeros(features)) # bias is a learnable parameter

    def forward(self, x):
        # x: (batch, seq_len, hidden_size)
        # keep the dimention for broadcasting
        mean = x.mean(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # keep the dimention for broadcasting
        std = x.std(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # eps is to prevent dividing by zeros or when std is very small
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

class FeedForwardBlock(nn.Module):

    def __init__(self, d_model: int, d_ff: int, dropout: float=0.1) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff) # w1 and b1
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model) # w2 and b2
    
    def forward(self, x):
        # (batch, seq_len, d_model) --> (batch, seq_len, d_ff) --> (batch, seq_len, d_model)
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))
    

class InputEmbeddings(nn.Module):

    def __init__(self, length: int, channels: int, max_timescale: float) -> None:
        super().__init__()
        self.length = length
        self.channels = channels
        self.max_timescale = max_timescale
        
    def sinusodis(self, x):
        """Returns sinusoids for positional embeddings."""
        assert self.channels % 2 == 0
        log_timescale_increment = np.log(self.max_timescale) / (self.channels // 2 - 1)
        inv_timescales = torch.exp(-log_timescale_increment * torch.arange(self.channels // 2))
        scaled_time = torch.arange(self.length)[:, np.newaxis] * inv_timescales[np.newaxis, :]
        return torch.cat()
